# Эксперименты и сравнение моделей

Здесь представлено сравнение моделей сначала на очищенных данных без feature engineering, затем на данных с добавленными признаками.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from joblib import dump
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.modeling import build_model_pipeline, evaluate_classifier, load_raw_data, split_data
from src.preprocessing import TARGET_COLUMN, clean_data, prepare_data

RANDOM_STATE = 42
MODEL_PATH = PROJECT_ROOT / 'models' / 'best_model_cp1.joblib'
METRICS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'metrics_cp1.csv'

In [2]:
raw_df = load_raw_data()
clean_df = clean_data(raw_df)
fe_df = prepare_data(raw_df)

clean_train, clean_val, clean_test = split_data(clean_df)
fe_train, fe_val, fe_test = split_data(fe_df)

pd.DataFrame(
    [
        {'dataset': 'raw_df', 'rows': raw_df.shape[0], 'cols': raw_df.shape[1]},
        {'dataset': 'clean_df', 'rows': clean_df.shape[0], 'cols': clean_df.shape[1]},
        {'dataset': 'fe_df', 'rows': fe_df.shape[0], 'cols': fe_df.shape[1]},
    ]
)

,dataset,rows,cols
0,raw_df,119390,32
1,clean_df,86638,30
2,fe_df,86380,38


## 1. Модели на данных без feature engineering

In [3]:
model_specs = [
    ('logistic_regression', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ('knn_classifier', KNeighborsClassifier(n_neighbors=15)),
    ('decision_tree', DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE)),
    ('random_forest', RandomForestClassifier(n_estimators=120, max_depth=14, n_jobs=1, random_state=RANDOM_STATE)),
    ('gradient_boosting', GradientBoostingClassifier(random_state=RANDOM_STATE)),
]

x_clean_train = clean_train.drop(columns=[TARGET_COLUMN])
y_clean_train = clean_train[TARGET_COLUMN]
x_clean_val = clean_val.drop(columns=[TARGET_COLUMN])
y_clean_val = clean_val[TARGET_COLUMN]

clean_rows = []
clean_models = {}

for model_name, estimator in model_specs:
    pipeline = build_model_pipeline(clean_train, estimator)
    pipeline.fit(x_clean_train, y_clean_train)
    metrics = evaluate_classifier(pipeline, x_clean_val, y_clean_val)
    clean_rows.append({'data_version': 'clean_data', 'model': model_name, **metrics})
    clean_models[model_name] = pipeline

In [4]:
clean_results = pd.DataFrame(clean_rows).sort_values('f1', ascending=False).reset_index(drop=True)
clean_results

,data_version,model,accuracy,precision,recall,f1,roc_auc
0,clean_data,gradient_boosting,0.828793,0.754731,0.565314,0.646433,0.889197
1,clean_data,decision_tree,0.810172,0.677217,0.600611,0.636618,0.869268
2,clean_data,knn_classifier,0.800939,0.675582,0.540578,0.600587,0.838760
3,clean_data,logistic_regression,0.796476,0.685625,0.489161,0.570965,0.843838
4,clean_data,random_forest,0.812327,0.817534,0.414675,0.550249,0.886436


## 2. Те же модели на данных с feature engineering

In [5]:
x_fe_train = fe_train.drop(columns=[TARGET_COLUMN])
y_fe_train = fe_train[TARGET_COLUMN]
x_fe_val = fe_val.drop(columns=[TARGET_COLUMN])
y_fe_val = fe_val[TARGET_COLUMN]

fe_rows = []
fe_models = {}

for model_name, estimator in model_specs:
    pipeline = build_model_pipeline(fe_train, estimator)
    pipeline.fit(x_fe_train, y_fe_train)
    metrics = evaluate_classifier(pipeline, x_fe_val, y_fe_val)
    fe_rows.append({'data_version': 'feature_engineering', 'model': model_name, **metrics})
    fe_models[model_name] = pipeline

In [6]:
fe_results = pd.DataFrame(fe_rows).sort_values('f1', ascending=False).reset_index(drop=True)
fe_results

,data_version,model,accuracy,precision,recall,f1,roc_auc
0,feature_engineering,gradient_boosting,0.835379,0.757587,0.589210,0.662873,0.894895
1,feature_engineering,decision_tree,0.815389,0.692384,0.590053,0.637136,0.870959
2,feature_engineering,knn_classifier,0.799028,0.669627,0.529643,0.591465,0.839172
3,feature_engineering,logistic_regression,0.796867,0.684148,0.483844,0.566820,0.852067
4,feature_engineering,random_forest,0.816084,0.821311,0.422310,0.557803,0.889401


## 3. Сводное сравнение

In [7]:
comparison = pd.concat([clean_results, fe_results], ignore_index=True)
comparison = comparison.sort_values(['f1', 'data_version'], ascending=[False, False]).reset_index(drop=True)
comparison

,data_version,model,accuracy,precision,recall,f1,roc_auc
0,feature_engineering,gradient_boosting,0.835379,0.757587,0.589210,0.662873,0.894895
1,clean_data,gradient_boosting,0.828793,0.754731,0.565314,0.646433,0.889197
2,feature_engineering,decision_tree,0.815389,0.692384,0.590053,0.637136,0.870959
3,clean_data,decision_tree,0.810172,0.677217,0.600611,0.636618,0.869268
4,clean_data,knn_classifier,0.800939,0.675582,0.540578,0.600587,0.838760
5,feature_engineering,knn_classifier,0.799028,0.669627,0.529643,0.591465,0.839172
6,clean_data,logistic_regression,0.796476,0.685625,0.489161,0.570965,0.843838
7,feature_engineering,logistic_regression,0.796867,0.684148,0.483844,0.566820,0.852067
8,feature_engineering,random_forest,0.816084,0.821311,0.422310,0.557803,0.889401
9,clean_data,random_forest,0.812327,0.817534,0.414675,0.550249,0.886436


In [8]:
comparison.pivot(index='model', columns='data_version', values='f1').sort_values('feature_engineering', ascending=False)

data_version,clean_data,feature_engineering
model,,
gradient_boosting,0.646433,0.662873
decision_tree,0.636618,0.637136
knn_classifier,0.600587,0.591465
logistic_regression,0.570965,0.566820
random_forest,0.550249,0.557803


feature engineering в целом оказался полезен, но не для всех моделей одинаково. Лучший результат по F1-score показал gradient_boosting, и после добавления новых признаков его качество выросло с 0.6464 до 0.6629.

Для decision_tree и random_forest улучшение почти незаметное: 0.6366 => 0.6371 у decision_tree и 0.5502 => 0.5578 у random_forest. Улучшение, хоть и небольшое, но есть.

У knn_classifier и logistic_regression качество после feature engineering немного снизилось. Разные модели по-разному чувствительны к структуре признакового пространства.

FE лучше всего сработал для бустинга, а для более простых или чувствительных моделей пользы не дал. Еще на этапе теста на простых очищенных данных gradient_boosting дал лучшие результаты, а с добавление FE его метрики увеличились. Несмотря на то, что метрики baseline модели стали хуже, я считаю, что feature engineering пошёл на пользу.

## 4. Выбор и сохранение лучшей модели

In [9]:
best_row = comparison.iloc[0]
best_row

data_version    feature_engineering
model             gradient_boosting
accuracy                   0.835379
precision                  0.757587
recall                      0.58921
f1                         0.662873
roc_auc                    0.894895
Name: 0, dtype: object

In [10]:
if best_row['data_version'] == 'feature_engineering':
    best_model = fe_models[best_row['model']]
else:
    best_model = clean_models[best_row['model']]

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(best_model, MODEL_PATH)
comparison.to_csv(METRICS_PATH, index=False)

MODEL_PATH, METRICS_PATH

(WindowsPath('C:/Users/user/Desktop/ml-project/hseml-group-project-ruf019/models/best_model_cp1.joblib'),
 WindowsPath('C:/Users/user/Desktop/ml-project/hseml-group-project-ruf019/data/processed/metrics_cp1.csv'))